# Домашнее задание 1

In [ ]:
import json
import os
import re
import subprocess
import sys
import time

from collections import Counter
from collections.abc import Callable, Collection, Iterable, Mapping
from concurrent.futures import ThreadPoolExecutor
from functools import lru_cache
from html import escape
from pathlib import Path
from statistics import mean
from time import perf_counter
from typing import Any, Self, cast
from urllib.request import urlretrieve

import httpx
import pandas as pd
import torch

from IPython.display import HTML, display
from openai import LengthFinishReasonError, OpenAI
from openai.types.chat import ChatCompletion, ChatCompletionMessageParam
from pydantic import (
    BaseModel,
    Field,
    ValidationError,
    field_validator,
    model_validator,
)

In [ ]:
print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA в PyTorch:", torch.version.cuda)
print("GPU доступен:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

!nvidia-smi

Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
PyTorch: 2.11.0+cu130
CUDA в PyTorch: 13.0
GPU доступен: True
GPU: Tesla T4
Sun Nov 15 14:37:51 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8             10W /   70W |       3MiB /  15360MiB |      0%      Default |
|    

In [3]:
%pip install -q -U uv "openai>=2.25,<3" "pydantic>=2,<3"

!uv venv --python {sys.executable} /tmp/hw1-vllm
!uv pip install --python /tmp/hw1-vllm/bin/python "vllm==0.31.0" --torch-backend=cu129

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.30.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
Note: you may need to restart the kernel to use updated packages.
Using CPython 3.13.16 interpreter at: /usr/bin/python3
Creating virtual environment at: /tmp/hw1-vllm
Activate with: source /tmp/hw1-vllm/bin/activate
Using Python 3.13.16 environment at: /tmp/hw1-vllm
Resolved 201 packages in 1.56s                                       
Prepared 201 packages in 1m 14s                                          
Installed 201 packages in 819ms                             
 + agent-detector==2.0.0
 + aiohappyeyeballs==2.7.1
 + aiohttp==3.14.5
 + aiosignal==1.4.0
 + annotated-doc==0.0.6
 + annotated-types==0.8.0
 + anthropic==2.8.0
 + anyio==4.16.0
 + apache-tvm-ffi==0.1.11
 + astor==0.8.1
 + attrs==26.1.0
 + blake3==1.0.11
 + ca

In [ ]:
MODEL = "Qwen/Qwen3-1.7B"
log_path = Path("/kaggle/working/vllm.log")
command = [
    "/tmp/hw1-vllm/bin/vllm", "serve", MODEL,
    "--host", "127.0.0.1", "--port", "8000",
    "--dtype", "half", "--max-model-len", "4096",
    "--max-num-seqs", "8", "--gpu-memory-utilization", "0.8",
    "--generation-config", "vllm", "--enforce-eager",
    "--structured-outputs-config", json.dumps({"backend": "guidance"}),
    "--reasoning-parser", "qwen3",
    "--reasoning-config", json.dumps({
        "reasoning_start_str": "<think>", "reasoning_end_str": "</think>"
    }),
]
if "server" in globals() and server.poll() is None and server.args != command:
    print("Перезапускаю vLLM с лимитом рассуждения...", flush=True)
    server.terminate()
    try:
        server.wait(timeout=30)
    except subprocess.TimeoutExpired:
        server.kill()
        server.wait()
if "server" not in globals() or server.poll() is not None:
    with log_path.open("w") as log:
        server = subprocess.Popen(
            command, env={**os.environ, "CUDA_VISIBLE_DEVICES": "0"},
            stdout=log, stderr=subprocess.STDOUT,
        )
for _ in range(120):
    if server.poll() is not None:
        raise RuntimeError(log_path.read_text(errors="replace")[-6000:])
    try:
        if httpx.get("http://127.0.0.1:8000/health", timeout=2).status_code == 200:
            print("Сервер готов. PID:", server.pid, "Backend: guidance")
            break
    except httpx.HTTPError:
        pass
    time.sleep(1)
else:
    raise TimeoutError(log_path.read_text(errors="replace")[-3000:])

Сервер готов. PID: 214 Backend: guidance


In [5]:
client = OpenAI(
    base_url="http://localhost:8000/v1",
    api_key="EMPTY",
    timeout=3600,
)
models = client.models.list()
MODEL = models.data[0].id
NO_THINK = {"chat_template_kwargs": {"enable_thinking": False}}
SAMPLING: dict[str, Any] = {
    "temperature": 0.7,
    "top_p": 0.8,
    "extra_body": {"top_k": 20},
}
RULES = (
    "По правилам скидка распределяется пропорционально стоимости товаров, доставка не является товаром, "
    "деньги за доставку не возвращаются."
)
ORDER = (
    "Наушники стоят 4000 руб., чехол 1000 руб. Скидка на товары 1000 руб., доставка 300 руб. "
    "Возвращаю только наушники."
)
HEADPHONES = {
    "input": ORDER,
    "answer": 3200,
    "returned_rub": 4000,
    "discount_rub": 1000,
    "delivery_rub": 300,
}
pd.set_option("display.max_colwidth", 250)

Message = Mapping[str, Any]


def chat(
    messages: Iterable[Message],
    think: bool = False,
    extra_body: dict[str, Any] | None = None,
    **params: Any,
) -> ChatCompletion:
    """Ответ модели целиком, вместе с usage; по умолчанию жадный выбор без рассуждения."""
    params = {
        "temperature": 0,
        "max_tokens": 1000,
        **params,
    }
    base = {} if think else NO_THINK
    extra_body = base | (extra_body or {})

    return client.chat.completions.create(
        model=MODEL,
        messages=cast(Iterable[ChatCompletionMessageParam], messages),
        extra_body=extra_body,
        **params,
    )


def structured[Schema: BaseModel](
    messages: Iterable[Message],
    schema: type[Schema],
    extra_body: dict[str, Any] | None = None,
    **params: Any,
) -> tuple[Schema | None, int]:
    """Ответ по схеме без рассуждения и число токенов; None, если генерация оборвалась на max_tokens."""
    params = {
        "temperature": 0,
        "max_tokens": 1000,
        **params,
    }
    try:
        response = client.chat.completions.parse(
            model=MODEL,
            messages=cast(Iterable[ChatCompletionMessageParam], messages),
            response_format=schema,
            extra_body=NO_THINK | (extra_body or {}),
            **params,
        )
    except LengthFinishReasonError as error:
        usage = error.completion.usage
        assert usage is not None

        return None, usage.completion_tokens

    assert response.usage is not None

    return response.choices[0].message.parsed, response.usage.completion_tokens


def parallel[Item, Result](fn: Callable[[Item], Result], items: Collection[Item]) -> list[Result]:
    """fn для каждого элемента; запросы идут параллельно."""
    with ThreadPoolExecutor(max(1, len(items))) as pool:
        results = pool.map(fn, items)

        return list(results)


def refund(instruction: str, order: str) -> list[dict[str, str]]:
    """Запрос на расчёт: в system -- инструкция и правила компании, в user -- заказ."""
    return [
        {
            "role": "system",
            "content": instruction + "\n\n" + RULES,
        },
        {
            "role": "user",
            "content": order,
        },
    ]


def read_jsonl(path: str) -> list[dict[str, Any]]:
    """Записи JSONL-файла."""
    lines = Path(path).read_text(encoding="utf-8").splitlines()

    return [json.loads(line) for line in lines]

## Задача 1 (10 баллов)

Реализуйте Tree of Thoughts для задачи рассчета суммы возврата. Рассуждение должно идти по шагам плана `PLAN`: на каждом шаге модель должна предложить несколько вариантов, оценщик оценить их на корректность и способность привести к решению задачи. В случае некорректности или если рассуждение зашло не туда -- вариант отбрасывается.

Критерии:

- +3 балла: на каждый шаг генерируется n различных вариантов;
- +3 балла: Оценщик делает не более двух ошибок для готовых `STATES`;
- +4 балла: Полная реализация алгоритма с генерацией вариантов для каждого шага, их оценкой и продолжением корректных шагов. 

In [6]:
PLAN = [
    "Стоимость товаров",
    "Скидка на возвращаемые товары",
    "Сумма возврата",
]

NO_THINK = {"chat_template_kwargs": {"enable_thinking": False}}

RULES = (
    "Скидка распределяется пропорционально стоимости товаров. "
    "Доставка не является товаром, деньги за доставку не возвращаются."
)

STEP_RULES = {
    "Стоимость товаров": (
        "Сумма исходных цен ВСЕХ товаров ДО скидки, без доставки. "
        "Скидку здесь не вычитай."
    ),
    "Скидка на возвращаемые товары": (
        "Общая скидка * исходная стоимость возвращаемых товаров / "
        "исходная стоимость всех товаров ДО скидки."
    ),
    "Сумма возврата": (
        "Исходная стоимость возвращаемых товаров МИНУС скидка на них. "
        "Доставку не прибавляй."
    ),
}

In [7]:
from functools import lru_cache
from time import perf_counter

class StepVariant(BaseModel):
    expression: str = Field(description="Арифметическое выражение для нужной величины: только числа, скобки и операции. Без знака равенства и единиц измерения.")
    value: float = Field(description="Числовой результат выражения в рублях")

@lru_cache(maxsize=128)
def _propose_cached(order: str, steps: tuple[str, ...], n: int) -> tuple[str, ...]:
    name = PLAN[len(steps)]
    class Batch(BaseModel):
        variants: list[StepVariant] = Field(min_length=n, max_length=n)
    variants = {}
    last_batch = None
    for attempt in range(4):
        print(f"Генерация: {name}, попытка {attempt + 1}", flush=True)
        instruction = (
            f'Предложи {n} разных способов вычислить только величину "{name}". '
            + STEP_RULES[name]
            + ' В каждом варианте expression содержит арифметическое выражение, '
            'а value — его результат. Выражения должны различаться, итоги могут совпадать. '
            'Используй перестановку слагаемых, скобки, дроби или эквивалентные преобразования. '
            'Не добавляй доставку и не меняй исходные данные. Без пояснений.'
        )
        context = order + "\nПринятые шаги:\n" + "\n".join(steps)
        if variants:
            context += "\nНе повторяй выражения:\n" + "\n".join(variants.keys())
        batch, _ = structured(
            refund(instruction, context), Batch,
            temperature=0.6, top_p=0.95, seed=attempt, max_tokens=1100,
            extra_body={"chat_template_kwargs": {"enable_thinking": True}, "thinking_token_budget": 512, "top_k": 20},
        )
        last_batch = batch
        if batch is None:
            continue
        for variant in batch.variants:
            expression = variant.expression.strip()
            if not re.fullmatch(r"[\d\s+*/().,·×−-]+", expression):
                continue
            key = re.sub(r"\s+", "", expression)
            key = key.replace("×", "*").replace("·", "*").replace("−", "-")
            variants[key] = f"{name}: {expression} = {variant.value:g} руб."
        if len(variants) >= n:
            return tuple(list(variants.values())[:n])
    raise RuntimeError(f"Получено {len(variants)} вариантов вместо {n}. Ответ: {last_batch}")

def propose(task: dict[str, Any], steps: list[str], n: int = 3) -> list[str]:
    if n <= 0 or len(steps) >= len(PLAN):
        return []
    return list(_propose_cached(task["input"], tuple(steps), n))

In [8]:
@lru_cache(maxsize=256)
def _evaluate_cached(order: str, steps: tuple[str, ...]) -> bool:
    instruction = (
        "Проверь все записанные шаги решения по исходному заказу. "
        "Сначала самостоятельно вычисли нужные величины, "
        "затем сравни с предложенными шагами. "
        "Проверь арифметику и смысл действий. "
        "Правильный начальный фрагмент плана можно продолжать: "
        "будущие шаги пока не требуются. "
        "Эквивалентные выражения допустимы. "
        "После рассуждения ответь только одной строкой: "
        "'Вердикт: продолжать' или 'Вердикт: отбросить'.\n"
        + "\n".join(f"{name}: {STEP_RULES[name]}" for name in PLAN)
    )
    context = order + "\nПроверяемые шаги:\n" + "\n".join(steps)
    response = chat(
        refund(instruction, context), think=True,
        temperature=0.6, top_p=0.95, seed=0, max_tokens=1200,
        extra_body={"chat_template_kwargs": {"enable_thinking": True}, "top_k": 20, "thinking_token_budget": 768},
    )
    choice = response.choices[0]
    if choice.finish_reason == "length":
        raise RuntimeError("Оценщик не закончил рассуждение в пределах лимита токенов")
    text = (choice.message.content or "").rsplit("</think>", 1)[-1]
    text = text.replace("**", "").strip().strip(chr(96) + '\"«»')
    verdicts = re.findall(
        r"^\s*Вердикт:\s*(продолжать|продолжить|отбросить|отклонить)\s*[.!]?\s*$",
        text, flags=re.IGNORECASE | re.MULTILINE,
    )
    if not verdicts:
        raise RuntimeError(f"Не распознан вердикт: {text[-500:]!r}")
    return verdicts[-1].casefold() in {"продолжать", "продолжить"}

def evaluate(task: dict[str, Any], steps: list[str]) -> bool:
    if not steps:
        return True
    if len(steps) > len(PLAN) or any(
        not step.startswith(name + ":") for name, step in zip(PLAN, steps)
    ):
        return False
    return _evaluate_cached(task["input"], tuple(steps))

def tree_of_thoughts(task: dict[str, Any], n: int = 3) -> list[str] | None:
    started = perf_counter()
    def search(steps):
        if len(steps) == len(PLAN):
            return steps
        variants = propose(task, steps, n)
        assert len(variants) == len(set(variants)) == n, "Нужны n разных вариантов"
        print(f"\nШаг {len(steps) + 1}: {len(variants)} вариантов", flush=True)
        print("\n".join(variants), flush=True)
        candidates = [steps + [variant] for variant in variants]
        print("Проверяю варианты параллельно...", flush=True)
        accepted = parallel(lambda candidate: evaluate(task, candidate), candidates)
        for candidate, ok in zip(candidates, accepted):
            print(f"{candidate[-1]} -> принят: {ok}", flush=True)
            if ok:
                result = search(candidate)
                if result is not None:
                    return result
        print("Ветка исчерпана, возврат к предыдущему шагу", flush=True)
        return None
    result = search([])
    print(f"Поиск: {perf_counter() - started:.1f} с", flush=True)
    return result

In [9]:
GOODS = "Стоимость товаров: 4000 + 1000 = 5000 руб."
SHARE = "Скидка на возвращаемые товары: 1000 · 4000 / 5000 = 800 руб."
STATES = [
    ([GOODS], True),
    (["Стоимость товаров: 4000 + 1000 + 300 = 5300 руб."], False),
    ([GOODS, SHARE], True),
    ([GOODS, "Скидка на возвращаемые товары: 1000 / 2 = 500 руб."], False),
    ([GOODS, SHARE, "Сумма возврата: 4000 − 800 = 3200 руб."], True),
    ([GOODS, SHARE, "Сумма возврата: 4000 + 800 = 4800 руб."], False),
    ([GOODS, "Сумма возврата: 4000 руб."], False),
]
_evaluate_cached.cache_clear()
started = perf_counter()
verdicts = parallel(lambda steps: evaluate(HEADPHONES, steps), [steps for steps, _ in STATES])
errors = sum(actual != expected for actual, (_, expected) in zip(verdicts, STATES))
print("Вердикты:", verdicts)
print("Ошибок оценщика:", errors)
print(f"Проверка оценщика: {perf_counter() - started:.1f} с")
assert errors <= 2, "Не выполнен критерий оценщика"
pd.DataFrame({
    "последний шаг": [steps[-1] for steps, _ in STATES],
    "по правилам": [expected for _, expected in STATES],
    "оценщик": verdicts,
})

Вердикты: [True, True, True, False, True, False, False]
Ошибок оценщика: 1
Проверка оценщика: 46.5 с


,последний шаг,по правилам,оценщик
0,Стоимость товаров: 4000 + 1000 = 5000 руб.,True,True
1,Стоимость товаров: 4000 + 1000 + 300 = 5300 руб.,False,True
2,Скидка на возвращаемые товары: 1000 · 4000 / 5000 = 800 руб.,True,True
3,Скидка на возвращаемые товары: 1000 / 2 = 500 руб.,False,False
4,Сумма возврата: 4000 − 800 = 3200 руб.,True,True
5,Сумма возврата: 4000 + 800 = 4800 руб.,False,False
6,Сумма возврата: 4000 руб.,False,False


In [39]:
solution = tree_of_thoughts(HEADPHONES, n=3)
assert solution is not None, "Поиск не нашёл решения"
assert len(solution) == len(PLAN), "Нужны все три шага"

amounts = []
for name, step in zip(PLAN, solution):
    assert step.startswith(name + ":"), "Нарушен порядок шагов"
    match = re.search(r"=\s*(-?\d+(?:[.,]\d+)?)\s*руб\.?\s*$", step)
    assert match is not None, f"Не распознана сумма: {step}"
    amounts.append(float(match.group(1).replace(",", ".")))
assert amounts == [5000, 800, HEADPHONES["answer"]], amounts

print("\nПолное решение:")
print("\n".join(solution))
print("Проверка всех трёх шагов пройдена")


Шаг 1: 3 вариантов
Стоимость товаров: 4000 + 1000 = 5000 руб.
Стоимость товаров: 1000 + 4000 = 5000 руб.
Стоимость товаров: 5000 - 0 = 5000 руб.
Проверяю варианты параллельно...
Стоимость товаров: 4000 + 1000 = 5000 руб. -> принят: True

Шаг 2: 3 вариантов
Скидка на возвращаемые товары: (1000 / 5000) * 4000 = 800 руб.
Скидка на возвращаемые товары: (4000 / 5000) * 1000 = 800 руб.
Скидка на возвращаемые товары: 1000 * (4000 / 5000) = 800 руб.
Проверяю варианты параллельно...
Скидка на возвращаемые товары: (1000 / 5000) * 4000 = 800 руб. -> принят: True

Шаг 3: 3 вариантов
Сумма возврата: 4000 - (1000 / 5000) * 4000 = 3200 руб.
Сумма возврата: 4000 - 1000 = 3000 руб.
Сумма возврата: 4000 * (1 - 1000 / 5000) = 3200 руб.
Проверяю варианты параллельно...
Сумма возврата: 4000 - (1000 / 5000) * 4000 = 3200 руб. -> принят: True
Поиск: 0.0 с

Полное решение:
Стоимость товаров: 4000 + 1000 = 5000 руб.
Скидка на возвращаемые товары: (1000 / 5000) * 4000 = 800 руб.
Сумма возврата: 4000 - (1000 / 

## Задача 2 (5 баллов)

Необходимо реализовать валидацию ответа LLM:

- номер заказа -- допустимы только цифры;
- цена, скидка и сумма возврата не отрицательные целые числа;
- сумма возврата не больше цены товара.

В `refund_requests.jsonl` -- обращения и ответы на них. Все ответы подходят под схему, но часть нарушает правила валидации. Напишите валидаторы, а потом промпт `EXTRACTOR`, с которым модель отвечает по правилам.

Критерии:

- +3 балла: по одному баллу за каждый валидатор
- +2 балла: доля корректных ответов модели не меньше 0.9 (+1 балл если меньше.).

In [ ]:
class RefundRequest(BaseModel):
    order_id: str
    item: str
    price_rub: int
    discount_rub: int
    refund_rub: int

    @field_validator("order_id")
    @classmethod
    def order_id_digits(cls, value: str) -> str:
        if re.fullmatch(r"[0-9]+", value) is None:
            raise ValueError("Номер заказа должен содержать только цифры")
        return value

    @field_validator("price_rub", "discount_rub", "refund_rub")
    @classmethod
    def not_negative(cls, value: int) -> int:
        if value < 0:
            raise ValueError("Сумма не может быть отрицательной")
        return value

    @model_validator(mode="after")
    def refund_within_price(self) -> Self:
        if self.refund_rub > self.price_rub:
            raise ValueError("Возврат не может превышать цену товара")
        return self

In [11]:
url = (
    "https://raw.githubusercontent.com/ozon-mlp-courses-org/"
    "ozon-llm-1-course/main/seminars/03-prompt-engineering/"
    "data/refund_requests.jsonl"
)

path = Path("/kaggle/working/data/refund_requests.jsonl")
path.parent.mkdir(parents=True, exist_ok=True)

urlretrieve(url, path)

(PosixPath('/kaggle/working/data/refund_requests.jsonl'),
 <http.client.HTTPMessage at 0x79e670fd6490>)

In [18]:
def is_valid(text: str) -> bool:
    """Проходит ли JSON-ответ проверку RefundRequest."""
    try:
        RefundRequest.model_validate_json(text)
    except ValidationError:
        return False

    return True


cases = read_jsonl("/kaggle/working/data/refund_requests.jsonl")
case_answers = [json.dumps(case["answer"], ensure_ascii=False) for case in cases]

pd.DataFrame({
    "обращение": [case["input"] for case in cases],
    "ответ": case_answers,
    "принят": [is_valid(answer) for answer in case_answers],
})

,обращение,ответ,принят
0,"Заказ 6120: мышь 1500 руб., бонусами списано −300. Хочу вернуть.","{""order_id"": ""{6120}"", ""item"": ""мышь"", ""price_rub"": 1500, ""discount_rub"": 300, ""refund_rub"": 1200}",False
1,Заказ 7421: наушники 4000 руб. со скидкой 800 руб. Верните наушники.,"{""order_id"": ""{\""order_id\"": 7421}"", ""item"": ""наушники"", ""price_rub"": 4000, ""discount_rub"": 800, ""refund_rub"": 3200}",False
2,"Заказ 7421: наушники 4000 руб., в чеке ""Скидка: −800,00"". Верните деньги.","{""order_id"": ""7421"", ""item"": ""наушники"", ""price_rub"": 4000, ""discount_rub"": -800, ""refund_rub"": 3200}",False
3,Заказ 4410: кабель за 500 руб. В личном кабинете сумма возврата: −500 руб.,"{""order_id"": ""4410"", ""item"": ""кабель"", ""price_rub"": 500, ""discount_rub"": 0, ""refund_rub"": -500}",False
4,"Чек по заказу 5310: ""Чехол 1 450,00 / Скидка 150,00 / Итого 1 300,00"". Возвращаю чехол.","{""order_id"": ""5310"", ""item"": ""чехол"", ""price_rub"": 450, ""discount_rub"": 150, ""refund_rub"": 1300}",False
5,Заказ 7421: наушники 4000 руб. и доставка 300 руб. Верните всё.,"{""order_id"": ""7421"", ""item"": ""наушники"", ""price_rub"": 4000, ""discount_rub"": 0, ""refund_rub"": 4300}",False
6,Заказ 9001: часы 12 000 руб. и подарочная упаковка 1000 руб. Верните 13 000 руб.,"{""order_id"": ""9001"", ""item"": ""часы"", ""price_rub"": 12000, ""discount_rub"": 0, ""refund_rub"": 13000}",False
7,"Заказ 2210: часы за 9000 руб., скидка по купону 1000 руб., доставка 300 руб. Верните часы.","{""order_id"": ""2210"", ""item"": ""часы"", ""price_rub"": 9000, ""discount_rub"": 1000, ""refund_rub"": 8000}",True
8,"Заказ 3304: клавиатура 3000 руб., скидка 10 %. Верните деньги.","{""order_id"": ""3304"", ""item"": ""клавиатура"", ""price_rub"": 3000, ""discount_rub"": 300, ""refund_rub"": 2700}",True
9,"Заказ №8812: колонка за 6000 руб. не включается, хочу вернуть.","{""order_id"": ""8812"", ""item"": ""колонка"", ""price_rub"": 6000, ""discount_rub"": 0, ""refund_rub"": 6000}",True


In [22]:
NAIVE = "Извлеки из обращения номер заказа, товар, его цену, скидку на него и сумму, которую нужно вернуть клиенту."

EXTRACTOR = """
Извлеки данные возвращаемого товара. Верни только JSON по схеме RefundRequest.
Заполняй поля по порядку:
1. order_id — номер заказа строкой только из цифр, без № и скобок.
2. item — название возвращаемого товара.
3. price_rub — исходная цена самого товара до скидки.
   Доставка и подарочная упаковка — отдельные услуги, их не учитывай.
   Пробел внутри суммы — разделитель тысяч: "2 450,00" означает 2450.
4. discount_rub — только явно указанная скидка, купон или списанные бонусы.
   ЕСЛИ СКИДКА ИЛИ БОНУСЫ НЕ УПОМЯНУТЫ, ОБЯЗАТЕЛЬНО ЗАПИШИ 0.
   Цена товара, доставка, упаковка и сумма, которую просят вернуть,
   не являются скидкой. Не придумывай скидку из этих сумм.
   "Сумма возврата: −..." — запись возврата, а не скидки: она не меняет discount_rub.
   Минус перед скидкой или списанными бонусами убери; процент переведи в рубли.
5. refund_rub — вычисли max(0, price_rub - discount_rub).
   Если скидки нет, refund_rub равен полной цене товара.
   Не прибавляй доставку и упаковку. Запрошенная клиентом сумма
   и знак минус перед записью возврата не меняют формулу.
Все денежные поля — неотрицательные целые числа.
Сумму после слов «верните» не копируй в refund_rub: вычисли возврат по цене товара и скидке.

Примеры:
Обращение: Заказ 123: лампа 2400 руб., доставка 200 руб., упаковка 100 руб. Верните 2700 руб.
Ответ: {"order_id":"123","item":"лампа","price_rub":2400,"discount_rub":0,"refund_rub":2400}
Обращение: Заказ 456: адаптер 350 руб. В кабинете сумма возврата: −350 руб.
Ответ: {"order_id":"456","item":"адаптер","price_rub":350,"discount_rub":0,"refund_rub":350}
Обращение: Заказ 789: коврик 900 руб., бонусами списано −90. Хочу вернуть.
Ответ: {"order_id":"789","item":"коврик","price_rub":900,"discount_rub":90,"refund_rub":810}
Обращение: Заказ 246: рюкзак 18 000 руб. и подарочная упаковка 1500 руб. Верните 19 500 руб.
Ответ: {"order_id":"246","item":"рюкзак","price_rub":18000,"discount_rub":0,"refund_rub":18000}
"""


def extract(text: str, prompt: str) -> RefundRequest | None:
    """Ответ модели по схеме RefundRequest; None, если он нарушает правила."""
    messages = [
        {"role": "system", "content": prompt},
        {"role": "user", "content": text},
    ]
    try:
        answer, _ = structured(messages, RefundRequest, max_tokens=200)
    except ValidationError:
        return None
    return answer


PROMPTS = {
    "простой промпт": NAIVE,
    "свой промпт": EXTRACTOR,
}
extraction: dict[str, list[Any]] = {"обращение": [case["input"] for case in cases]}
for name, prompt in PROMPTS.items():
    extracted = parallel(lambda case: extract(case["input"], prompt), cases)
    accepted = [answer is not None for answer in extracted]
    extraction[name] = extracted
    print(f"{name}: доля принятых {mean(accepted):.1f}")
pd.DataFrame(extraction)

простой промпт: доля принятых 0.9
свой промпт: доля принятых 1.0


,обращение,простой промпт,свой промпт
0,"Заказ 6120: мышь 1500 руб., бонусами списано −300. Хочу вернуть.",order_id='6120' item='мышь' price_rub=1500 discount_rub=300 refund_rub=1200,order_id='6120' item='мышь' price_rub=1500 discount_rub=300 refund_rub=1200
1,Заказ 7421: наушники 4000 руб. со скидкой 800 руб. Верните наушники.,order_id='7421' item='наушники' price_rub=4000 discount_rub=800 refund_rub=3200,order_id='7421' item='наушники' price_rub=4000 discount_rub=800 refund_rub=3200
2,"Заказ 7421: наушники 4000 руб., в чеке ""Скидка: −800,00"". Верните деньги.",order_id='7421' item='наушники' price_rub=4000 discount_rub=800 refund_rub=3200,order_id='7421' item='наушники' price_rub=4000 discount_rub=800 refund_rub=3200
3,Заказ 4410: кабель за 500 руб. В личном кабинете сумма возврата: −500 руб.,None,order_id='4410' item='кабель' price_rub=500 discount_rub=0 refund_rub=500
4,"Чек по заказу 5310: ""Чехол 1 450,00 / Скидка 150,00 / Итого 1 300,00"". Возвращаю чехол.",order_id='5310' item='Чехол' price_rub=1450 discount_rub=150 refund_rub=1300,order_id='5310' item='Чехол' price_rub=1450 discount_rub=150 refund_rub=1300
5,Заказ 7421: наушники 4000 руб. и доставка 300 руб. Верните всё.,order_id='7421' item='наушники' price_rub=4000 discount_rub=4000 refund_rub=0,order_id='7421' item='наушники' price_rub=4000 discount_rub=0 refund_rub=4000
6,Заказ 9001: часы 12 000 руб. и подарочная упаковка 1000 руб. Верните 13 000 руб.,order_id='9001' item='часы' price_rub=12000 discount_rub=13000 refund_rub=1000,order_id='9001' item='часы' price_rub=12000 discount_rub=0 refund_rub=12000
7,"Заказ 2210: часы за 9000 руб., скидка по купону 1000 руб., доставка 300 руб. Верните часы.",order_id='2210' item='часы' price_rub=9000 discount_rub=1000 refund_rub=8000,order_id='2210' item='часы' price_rub=9000 discount_rub=1000 refund_rub=8000
8,"Заказ 3304: клавиатура 3000 руб., скидка 10 %. Верните деньги.",order_id='3304' item='клавиатура' price_rub=3000 discount_rub=300 refund_rub=2700,order_id='3304' item='клавиатура' price_rub=3000 discount_rub=300 refund_rub=2700
9,"Заказ №8812: колонка за 6000 руб. не включается, хочу вернуть.",order_id='8812' item='колонка' price_rub=6000 discount_rub=0 refund_rub=6000,order_id='8812' item='колонка' price_rub=6000 discount_rub=0 refund_rub=6000


## Задача 3 (5 баллов)

Необходимо решить задачу расчёта суммы возврата через Structured Outputs. Сумма должна возвращаться в поле `refund_rub`.

Напишите схему `Solution` и промпт `SOLVER`: перед тем как заполнить `refund_rub` модель должна выписать дополнительные вычисления. Учтите, что поля модель генерирует по порядку.

Постарайтесь решить без включения Reasoning.

Критерии:

- +1 балл: для всех кейсов модель генерирует валидный ответ
- +1 балл: кроме поля `refund_rub` -- в схеме есть ещё поля для промежуточных вычислений;
- +1 балл: в промпте нет правильных ответов из `TASKS`.
- +1 балл: доля правильных ответов больше 0.5
- +1 балл: Не использован Reasoning

In [ ]:
orders_path = Path("/kaggle/working/data/refund_orders.jsonl")
orders_path.parent.mkdir(parents=True, exist_ok=True)
if not orders_path.exists():
    urlretrieve(
        "https://raw.githubusercontent.com/ozon-mlp-courses-org/"
        "ozon-llm-1-course/main/seminars/03-prompt-engineering/data/refund_orders.jsonl",
        orders_path,
    )

TASKS = read_jsonl(str(orders_path))
print("Заказов для проверки:", len(TASKS))
pd.DataFrame(TASKS)[["input", "answer"]]

Заказов для проверки: 10


,input,answer
0,"В заказе клавиатура за 4200 рублей, чехол за 5400 рублей. Общая скидка на товары -- 400 рублей. Доставка -- 700 рублей. Возвращаемые товары: клавиатура.",4025
1,"В заказе колонка за 1300 рублей, зарядное устройство за 2100 рублей, часы за 3400 рублей. Общая скидка на товары -- 1700 рублей. Доставка -- 300 рублей. Возвращаемые товары: зарядное устройство.",1575
2,"В заказе наушники за 3900 рублей, чехол за 5700 рублей. Общая скидка на товары -- 1600 рублей. Доставка -- 200 рублей. Возвращаемые товары: наушники.",3250
3,"В заказе клавиатура за 6400 рублей, часы за 4600 рублей. Общая скидка на товары -- 2200 рублей. Доставка -- 200 рублей. Возвращаемые товары: клавиатура.",5120
4,"В заказе мышь за 7100 рублей, зарядное устройство за 7100 рублей. Общая скидка на товары -- 2800 рублей. Доставка -- 500 рублей. Возвращаемые товары: зарядное устройство.",5700
5,"В заказе часы за 4500 рублей, зарядное устройство за 4500 рублей. Общая скидка на товары -- 800 рублей. Доставка -- 300 рублей. Возвращаемые товары: часы.",4100
6,"В заказе зарядное устройство за 5200 рублей, часы за 6000 рублей, колонка за 2800 рублей. Общая скидка на товары -- 1400 рублей. Доставка -- 300 рублей. Возвращаемые товары: колонка.",2520
7,"В заказе колонка за 4400 рублей, часы за 5600 рублей. Общая скидка на товары -- 400 рублей. Доставка -- 500 рублей. Возвращаемые товары: часы.",5376
8,"В заказе часы за 2200 рублей, чехол за 1800 рублей. Общая скидка на товары -- 2300 рублей. Доставка -- 700 рублей. Возвращаемые товары: чехол.",765
9,"В заказе кабель за 4200 рублей, зарядное устройство за 5300 рублей. Общая скидка на товары -- 1900 рублей. Доставка -- 400 рублей. Возвращаемые товары: зарядное устройство.",4240


In [13]:
class Solution(BaseModel):
    total_calculation: str = Field(description="Короткое сложение исходных цен ВСЕХ товаров до скидки, без доставки, с результатом")
    total_rub: int = Field(description="Сумма исходных цен всех товаров до скидки, без доставки")
    returned_rub: int = Field(description="Сумма исходных цен только возвращаемых товаров до скидки")
    order_discount_rub: int = Field(description="Общая скидка на весь заказ из условия, положительное число")
    discount_calculation: str = Field(description="Числовое выражение: общая скидка * стоимость возвращаемых товаров / стоимость всех товаров = скидка на возвращаемые товары")
    returned_discount_rub: float = Field(description="Результат discount_calculation: только доля общей скидки, приходящаяся на возвращаемые товары")
    refund_calculation: str = Field(description="Числовое вычитание: returned_rub - returned_discount_rub = сумма возврата")
    refund_rub: int = Field(description="Итог refund_calculation в рублях, без доставки")


SOLVER = """
Рассчитай сумму возврата по исходному заказу. Верни JSON по схеме Solution.
Заполняй поля строго по порядку. В полях calculation пиши короткое
арифметическое выражение с подставленными числами и вычисленным результатом.

Сначала сложи исходные цены ВСЕХ перечисленных товаров ДО скидки:
запиши сложение в total_calculation и результат в total_rub.
Доставку в эту сумму не включай. Скидку на этом этапе не вычитай.
В returned_rub запиши исходную стоимость только товаров, названных
после слов «Возвращаемые товары». В order_discount_rub перенеси общую скидку.

Скидка распределяется пропорционально исходным ценам товаров.
В discount_calculation подставь числа и вычисли:
order_discount_rub * returned_rub / total_rub.
Именно этот результат запиши в returned_discount_rub.
Общая скидка относится ко всему заказу: при частичном возврате
используй только её вычисленную долю для возвращаемых товаров.

В refund_calculation выполни returned_rub - returned_discount_rub.
Последним заполни refund_rub результатом этого вычитания.
Деньги за доставку не возвращаются. Не прибавляй доставку ни на одном этапе.
Проверь арифметику каждого выражения. Ответ — только JSON без текста вне схемы.
"""

In [14]:
def solve_structured[Schema: BaseModel](
    task: dict[str, Any],
    schema: type[Schema],
    instruction: str,
    **params: Any,
) -> tuple[Schema | None, int]:
    """Решение по схеме и число токенов; None, если ответ не разобран."""
    messages = refund(instruction, task["input"])
    return structured(messages, schema, **params)


def refund_of(answer: BaseModel | None) -> int | None:
    """Сумма возврата из разобранного ответа."""
    return getattr(answer, "refund_rub", None)


assert NO_THINK["chat_template_kwargs"]["enable_thinking"] is False
solutions = parallel(
    lambda task: solve_structured(task, Solution, SOLVER, temperature=0),
    TASKS,
)
refunds = [refund_of(answer) for answer, _ in solutions]
unparsed = sum(answer is None for answer, _ in solutions)
right = sum(refund == task["answer"] for refund, task in zip(refunds, TASKS))
leaked = [
    task["answer"] for task in TASKS
    if re.search(rf"(?<!\d){task['answer']}(?!\d)", SOLVER)
]

print("Поля Solution:", " -> ".join(Solution.model_fields))
print(f"Валидных ответов: {len(TASKS) - unparsed}/{len(TASKS)}")
print(f"Правильных сумм: {right}/{len(TASKS)} ({right / len(TASKS):.0%})")
print("Правильные ответы в промпте:", leaked or "нет")
print("Reasoning выключен:", NO_THINK["chat_template_kwargs"]["enable_thinking"] is False)

assert unparsed == 0, "Не все ответы разобраны по схеме"
assert not leaked, "В промпте обнаружены эталонные суммы"
assert right / len(TASKS) > 0.5, "Нужно строго больше половины правильных сумм"

pd.DataFrame({
    "задача": [task["input"] for task in TASKS],
    "эталон": [task["answer"] for task in TASKS],
    "ответ": refunds,
    "верно": [refund == task["answer"] for refund, task in zip(refunds, TASKS)],
})

Поля Solution: total_calculation -> total_rub -> returned_rub -> order_discount_rub -> discount_calculation -> returned_discount_rub -> refund_calculation -> refund_rub
Валидных ответов: 10/10
Правильных сумм: 10/10 (100%)
Правильные ответы в промпте: нет
Reasoning выключен: True


,задача,эталон,ответ,верно
0,"В заказе клавиатура за 4200 рублей, чехол за 5400 рублей. Общая скидка на товары -- 400 рублей. Доставка -- 700 рублей. Возвращаемые товары: клавиатура.",4025,4025,True
1,"В заказе колонка за 1300 рублей, зарядное устройство за 2100 рублей, часы за 3400 рублей. Общая скидка на товары -- 1700 рублей. Доставка -- 300 рублей. Возвращаемые товары: зарядное устройство.",1575,1575,True
2,"В заказе наушники за 3900 рублей, чехол за 5700 рублей. Общая скидка на товары -- 1600 рублей. Доставка -- 200 рублей. Возвращаемые товары: наушники.",3250,3250,True
3,"В заказе клавиатура за 6400 рублей, часы за 4600 рублей. Общая скидка на товары -- 2200 рублей. Доставка -- 200 рублей. Возвращаемые товары: клавиатура.",5120,5120,True
4,"В заказе мышь за 7100 рублей, зарядное устройство за 7100 рублей. Общая скидка на товары -- 2800 рублей. Доставка -- 500 рублей. Возвращаемые товары: зарядное устройство.",5700,5700,True
5,"В заказе часы за 4500 рублей, зарядное устройство за 4500 рублей. Общая скидка на товары -- 800 рублей. Доставка -- 300 рублей. Возвращаемые товары: часы.",4100,4100,True
6,"В заказе зарядное устройство за 5200 рублей, часы за 6000 рублей, колонка за 2800 рублей. Общая скидка на товары -- 1400 рублей. Доставка -- 300 рублей. Возвращаемые товары: колонка.",2520,2520,True
7,"В заказе колонка за 4400 рублей, часы за 5600 рублей. Общая скидка на товары -- 400 рублей. Доставка -- 500 рублей. Возвращаемые товары: часы.",5376,5376,True
8,"В заказе часы за 2200 рублей, чехол за 1800 рублей. Общая скидка на товары -- 2300 рублей. Доставка -- 700 рублей. Возвращаемые товары: чехол.",765,765,True
9,"В заказе кабель за 4200 рублей, зарядное устройство за 5300 рублей. Общая скидка на товары -- 1900 рублей. Доставка -- 400 рублей. Возвращаемые товары: зарядное устройство.",4240,4240,True


In [ ]:
blocks = []
for number, (task, (answer, tokens)) in enumerate(zip(TASKS, solutions), start=1):
    text = answer.model_dump_json(indent=2) if answer is not None else "Ответ не разобран"
    amount = refund_of(answer)
    opened = " open" if number == 1 else ""
    blocks.append(
        f'<details{opened} style="margin: 12px 0">'
        f'<summary><b>Ответ {number}</b>: возврат {amount} руб. '
        f'(эталон {task["answer"]} руб.)</summary>'
        f'<p>{escape(task["input"])}</p>'
        f'<pre>{escape(text)}</pre>'
        f'<p>Сгенерировано токенов: {tokens}</p></details>'
    )

display(HTML("".join(blocks)))

## Задача 4 (5 баллов)

Необходимо сравнить решение задачи 3 с базовым вариантом `Baseline`, в котором схема состоит только из поля `refund_rub`.

Напишите функцию `diagnose`, которая по сумме возврата и данным заказа определяет тип ответа:

- "верно" -- сумма совпала с правильной;
- "скидка вычтена целиком" -- из стоимости возвращаемых товаров вычтена вся скидка заказа;
- "скидка не вычтена" -- возвращена полная стоимость возвращаемых товаров;
- "доставка в сумме" -- к правильной сумме прибавлена доставка;
- "нет ответа" -- ответ не разобран;
- "другое" -- остальные ошибки.

Затем запустите оба варианта на `TASKS` с жадным выбором и с параметрами из `RUNS` и соберите таблицу: доля правильных ответов при жадном выборе, среднее, минимум и максимум доли по `RUNS`, токены на задачу и число ответов каждого типа.

Критерии:

- +3 балла: `diagnose` верно определяет тип всех ответов из `TESTS`;
- +2 балла: таблица сравнения для обоих вариантов;

In [16]:
def diagnose(task: dict[str, Any], answer: int | None) -> str:
    if answer is None:
        return "нет ответа"
    if answer == task["answer"]:
        return "верно"
    if answer == task["returned_rub"] - task["discount_rub"]:
        return "скидка вычтена целиком"
    if answer == task["returned_rub"]:
        return "скидка не вычтена"
    if answer == task["answer"] + task["delivery_rub"]:
        return "доставка в сумме"
    return "другое"


TESTS = [
    (3200, "верно"),
    (3000, "скидка вычтена целиком"),
    (4000, "скидка не вычтена"),
    (3500, "доставка в сумме"),
    (None, "нет ответа"),
    (2500, "другое"),
]

diagnostics = pd.DataFrame({
    "ответ": [answer for answer, _ in TESTS],
    "ожидается": [kind for _, kind in TESTS],
    "diagnose": [diagnose(HEADPHONES, answer) for answer, _ in TESTS],
})
assert (diagnostics["ожидается"] == diagnostics["diagnose"]).all()
diagnostics

,ответ,ожидается,diagnose
0,3200.0,верно,верно
1,3000.0,скидка вычтена целиком,скидка вычтена целиком
2,4000.0,скидка не вычтена,скидка не вычтена
3,3500.0,доставка в сумме,доставка в сумме
4,NaN,нет ответа,нет ответа
5,2500.0,другое,другое


In [ ]:
class Baseline(BaseModel):
    refund_rub: int


BASELINE = "Рассчитай сумму возврата за возвращаемые товары в рублях."
SAMPLING = {
    "temperature": 0.7,
    "top_p": 0.8,
    "extra_body": {"top_k": 20},
}
RUNS = {
    f"seed {seed}": {**SAMPLING, "seed": seed}
    for seed in range(3)
}
CONFIGS = {"жадный": {"temperature": 0}, **RUNS}
VARIANTS = {
    "Baseline": (Baseline, BASELINE),
    "Solution": (Solution, SOLVER),
}
KINDS = [
    "верно",
    "скидка вычтена целиком",
    "скидка не вычтена",
    "доставка в сумме",
    "нет ответа",
    "другое",
]

assert NO_THINK["chat_template_kwargs"]["enable_thinking"] is False
assert len(solutions) == len(TASKS), "Сначала запусти задачу 3"

records = []
task4_outputs = {}
for name, (schema, prompt) in VARIANTS.items():
    for run_name, params in CONFIGS.items():
        print(f"{name}: {run_name}", flush=True)

        if name == "Solution" and run_name == "жадный":
            batch = solutions
        else:
            batch = parallel(
                lambda task: solve_structured(task, schema, prompt, **params),
                TASKS,
            )

        task4_outputs[name, run_name] = batch
        for index, (task, (parsed, tokens)) in enumerate(zip(TASKS, batch)):
            amount = refund_of(parsed)
            records.append({
                "вариант": name,
                "запуск": run_name,
                "заказ": index,
                "ответ": amount,
                "эталон": task["answer"],
                "верно": amount == task["answer"],
                "токены": tokens,
                "тип": diagnose(task, amount),
            })

        accuracy = mean(
            refund_of(answer) == task["answer"]
            for task, (answer, _) in zip(TASKS, batch)
        )
        print(f"  правильных: {accuracy:.0%}", flush=True)

details = pd.DataFrame(records)
summary = []
for name in VARIANTS:
    rows = details[details["вариант"] == name]
    sampled = rows[rows["запуск"] != "жадный"]
    accuracies = sampled.groupby("запуск")["верно"].mean()
    counts = Counter(rows["тип"])

    summary.append({
        "вариант": name,
        "жадный: точность": rows.loc[
            rows["запуск"] == "жадный", "верно"
        ].mean(),
        "RUNS: средняя": accuracies.mean(),
        "RUNS: минимум": accuracies.min(),
        "RUNS: максимум": accuracies.max(),
        "токенов/задачу": rows["токены"].mean(),
        **{kind: counts[kind] for kind in KINDS},
    })

comparison = pd.DataFrame(summary).set_index("вариант")
print("Токены и типы ответов: все 4 режима, по 40 ответов на вариант.")
comparison

Baseline: жадный
  правильных: 0%
Baseline: seed 0
  правильных: 0%
Baseline: seed 1
  правильных: 0%
Baseline: seed 2
  правильных: 0%
Solution: жадный
  правильных: 100%
Solution: seed 0
  правильных: 100%
Solution: seed 1
  правильных: 100%
Solution: seed 2
  правильных: 100%
Токены и типы ответов: все 4 режима, по 40 ответов на вариант.


,жадный: точность,RUNS: средняя,RUNS: минимум,RUNS: максимум,токенов/задачу,верно,скидка вычтена целиком,скидка не вычтена,доставка в сумме,нет ответа,другое
вариант,,,,,,,,,,,
Baseline,0.0,0.0,0.0,0.0,38.075,0,7,10,0,1,22
Solution,1.0,1.0,1.0,1.0,125.100,40,0,0,0,0,0
